# Ingest Data
Ingestion includes:
- Defines the schema for each entity
- Writes into BRONZE tables  

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, ArrayType
from pyspark.sql.functions import col, current_timestamp, date_format, from_utc_timestamp

### Ingest `playback` files

In [0]:
# define the type of schema 
playback_schema = StructType([StructField("album_id", StringType(), True), 
                              StructField("track_id", StringType(), False), # track_id must not be null
                              StructField("played_at", StringType(), True)
                            ])


In [0]:
# autoloader 
playback_df = (
    spark.readStream
        .format("cloudFiles")
        .schema(playback_schema)
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv") # parse only .csv files in the directory
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/playback/schema")
        .option("cloudFiles.schemaEvolutionMode", "rescue") # stream will not fail when new columns detected, but rather added to the "rescue" column
        .load("/Volumes/spotify_dev/00_landing/spotify_operational_data/playback")
)

# write data to delta table - spotify_dev.00_bronze
(
    playback_df
    .withColumn("file_source", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"))
    .writeStream
        .option("mergeSchema", "true")
        .option("checkpointLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/playback/checkpoints")
        .outputMode("append")
        .trigger(availableNow=True) # process this structured streaming in batch mode -> process all the files that are present, then shut down
        .toTable("spotify_dev.01_bronze.playback") # save as a table
)

In [0]:
%sql
SELECT *
FROM spotify_dev.`01_bronze`.playback

In [0]:
%sql
-- DROP TABLE spotify_dev.`01_bronze`.playback

### Ingest `artist` files

In [0]:
# define the type of schema 
artist_schema = StructType([StructField("id", StringType(), False),
                            StructField("name", StringType(), True),
                            StructField("genres", StringType(), True)
                            ])


In [0]:
# autoloader 
artist_df = (
    spark.readStream
        .format("cloudFiles")
        .schema(artist_schema)
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv") # parse only .csv files in the directory
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/artist/schema")
        .option("cloudFiles.schemaEvolutionMode", "rescue") # stream will not fail when new columns detected, but rather added to the "rescue" column
        .load("/Volumes/spotify_dev/00_landing/spotify_operational_data/artist")
)

# write data to delta table - spotify_dev.00_bronze
(
    artist_df
    .withColumn("file_source", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"))
    .writeStream
        .option("mergeSchema", "true")
        .option("checkpointLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/artist/checkpoints")
        .outputMode("append")
        .trigger(availableNow=True) # process this structured streaming in batch mode -> process all the files that are present, then shut down
        .toTable("spotify_dev.01_bronze.artist") # save as a table
)

In [0]:
%sql
SELECT DISTINCT(*)
FROM spotify_dev.`01_bronze`.artist

### Ingest `album` files

In [0]:
# define the type of schema 
album_schema = StructType([StructField("id", StringType(), False),
                            StructField("name", StringType(), True),
                            StructField("album_type", StringType(), True),
                            StructField("release_date", DateType(), True),
                            StructField("total_tracks", IntegerType(), True)
                            ])


In [0]:
# autoloader 
album_df = (
    spark.readStream
        .format("cloudFiles")
        .schema(album_schema)
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv") # parse only .csv files in the directory
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/album/schema")
        .option("cloudFiles.schemaEvolutionMode", "rescue") # stream will not fail when new columns detected, but rather added to the "rescue" column
        .load("/Volumes/spotify_dev/00_landing/spotify_operational_data/artist")
)

# write data to delta table - spotify_dev.00_bronze
(
    album_df
    .withColumn("file_source", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"))
    .writeStream
        .option("mergeSchema", "true")
        .option("checkpointLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/album/checkpoints")
        .outputMode("append")
        .trigger(availableNow=True) # process this structured streaming in batch mode -> process all the files that are present, then shut down
        .toTable("spotify_dev.01_bronze.album") # save as a table
)

### Ingest `track` files

In [0]:
# define the type of schema 
track_schema = StructType([StructField("id", StringType(), False),
                            StructField("name", StringType(), True),
                            StructField("album", StringType(), True), # album entity - nested json
                            StructField("artists", StringType(), True), # array of artists entity
                            StructField("popularity", IntegerType(), True),
                            StructField("duration_ms", IntegerType(), True)
                            ])


In [0]:
# autoloader 
track_df = (
    spark.readStream
        .format("cloudFiles")
        .schema(track_schema)
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv") # parse only .csv files in the directory
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/track/schema")
        .option("cloudFiles.schemaEvolutionMode", "rescue") # stream will not fail when new columns detected, but rather added to the "rescue" column
        .load("/Volumes/spotify_dev/00_landing/spotify_operational_data/track")
)

# write data to delta table - spotify_dev.00_bronze
(
    track_df
    .withColumn("file_source", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"))
    .writeStream
        .option("mergeSchema", "true")
        .option("checkpointLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/autoloader/track/checkpoints")
        .outputMode("append")
        .trigger(availableNow=True) # process this structured streaming in batch mode -> process all the files that are present, then shut down
        .toTable("spotify_dev.01_bronze.track") # save as a table
)

In [0]:
%sql
SELECT *
FROM spotify_dev.`01_bronze`.track
LIMIT 50